In [1]:
import pandas as pd

df = pd.read_csv("retail_store_sales.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (12575, 11)


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  object 
 1   Customer ID       12575 non-null  object 
 2   Category          12575 non-null  object 
 3   Item              11362 non-null  object 
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  object 
 8   Location          12575 non-null  object 
 9   Transaction Date  12575 non-null  object 
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(8)
memory usage: 1.1+ MB


In [3]:
df.isnull().sum()

,0
Transaction ID,0
Customer ID,0
Category,0
Item,1213
Price Per Unit,609
Quantity,604
Total Spent,604
Payment Method,0
Location,0
Transaction Date,0


In [4]:
price_recovery = df[
    df["Price Per Unit"].isna() &
    df["Quantity"].notna() &
    df["Total Spent"].notna()
]

print("Recoverable Price rows:", len(price_recovery))

Recoverable Price rows: 609


In [5]:
df.loc[df["Price Per Unit"].isna(), "Price Per Unit"] = (
    df["Total Spent"] / df["Quantity"]
)

In [6]:
df["Price Per Unit"].isnull().sum()

np.int64(0)

In [7]:
quantity_recovery = df[
    df["Quantity"].isna() &
    df["Price Per Unit"].notna() &
    df["Total Spent"].notna()
]

print("Recoverable Quantity rows:", len(quantity_recovery))

Recoverable Quantity rows: 0


In [8]:
total_recovery = df[
    df["Total Spent"].isna() &
    df["Price Per Unit"].notna() &
    df["Quantity"].notna()
]

print("Recoverable Total Spent rows:", len(total_recovery))

Recoverable Total Spent rows: 0


In [9]:
df[df["Item"].isna()]["Category"].value_counts()

,count
Category,
Food,162
Computers and electric accessories,161
Patisserie,159
Milk Products,159
Electric household essentials,154
Butchers,147
Beverages,140
Furniture,131


In [10]:
item_check = df.dropna(
    subset=["Item", "Price Per Unit"]
)

item_check.groupby(
    ["Category", "Price Per Unit"]
)["Item"].nunique().max()


1

In [11]:
item_recovery = df[
    df["Item"].isna() &
    df["Price Per Unit"].notna()
]

print("Recoverable Item rows:", len(item_recovery))

Recoverable Item rows: 1213


In [12]:
item_map = (
    df.dropna(subset=["Item", "Price Per Unit"])
      .drop_duplicates(["Category", "Price Per Unit"])
      .set_index(["Category", "Price Per Unit"])["Item"]
)

In [13]:
df["Item"] = df["Item"].fillna(
    df.apply(
        lambda row: item_map.get(
            (row["Category"], row["Price Per Unit"])
        ),
        axis=1
    )
)

In [14]:
print("Missing Item values:", df["Item"].isnull().sum())

Missing Item values: 0


In [15]:
df["Discount Applied"].value_counts(dropna=False)

,count
Discount Applied,
True,4219
NaN,4199
False,4157


In [16]:
df["Discount Applied"] = df["Discount Applied"].fillna("Unknown")

In [17]:
incomplete_rows = df[
    df["Quantity"].isna() &
    df["Total Spent"].isna()
]

print("Incomplete transaction rows:", len(incomplete_rows))

Incomplete transaction rows: 604


In [18]:
df_clean = df.dropna(
    subset=["Quantity", "Total Spent"]
).copy()

print("Cleaned Dataset Shape:", df_clean.shape)

Cleaned Dataset Shape: (11971, 11)


In [19]:
df_clean.isnull().sum()

,0
Transaction ID,0
Customer ID,0
Category,0
Item,0
Price Per Unit,0
Quantity,0
Total Spent,0
Payment Method,0
Location,0
Transaction Date,0


In [20]:
print("Duplicate rows:", df_clean.duplicated().sum())

Duplicate rows: 0


In [21]:
df_clean["Transaction Date"] = pd.to_datetime(
    df_clean["Transaction Date"]
)

print(df_clean["Transaction Date"].dtype)

datetime64[ns]


In [22]:
calculated_total = (
    df_clean["Price Per Unit"] * df_clean["Quantity"]
)

difference = (
    df_clean["Total Spent"] - calculated_total
).abs()

print("Maximum difference:", difference.max())

Maximum difference: 0.0


In [23]:
print(df_clean.dtypes)

Transaction ID              object
Customer ID                 object
Category                    object
Item                        object
Price Per Unit             float64
Quantity                   float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
Discount Applied            object
dtype: object


In [24]:
df_clean.to_csv(
    "retail_store_sales_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [25]:
import pandas as pd

df_clean = pd.read_csv("retail_store_sales_cleaned.csv")

print("Dataset Shape:", df_clean.shape)
df_clean.head()

Dataset Shape: (11971, 11)


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,Unknown
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [26]:
category_groups = df_clean.groupby("Category")

print(category_groups)

In [27]:
category_sales = df_clean.groupby("Category")["Total Spent"].sum()

category_sales

,Total Spent
Category,
Beverages,197047.5
Butchers,208118.0
Computers and electric accessories,190692.5
Electric household essentials,203813.5
Food,194812.0
Furniture,195310.0
Milk Products,180112.0
Patisserie,182165.5


In [28]:
category_avg_sales = df_clean.groupby("Category")["Total Spent"].mean()

category_avg_sales

,Total Spent
Category,
Beverages,131.716243
Butchers,139.116310
Computers and electric accessories,129.107989
Electric household essentials,134.441623
Food,129.271400
Furniture,128.072131
Milk Products,119.042961
Patisserie,126.416031


In [29]:
category_count = df_clean.groupby("Category")["Transaction ID"].count()

category_count

,Transaction ID
Category,
Beverages,1496
Butchers,1496
Computers and electric accessories,1477
Electric household essentials,1516
Food,1507
Furniture,1525
Milk Products,1513
Patisserie,1441


In [30]:
payment_count = df_clean.groupby("Payment Method")["Transaction ID"].count()

payment_count

,Transaction ID
Payment Method,
Cash,4103
Credit Card,3927
Digital Wallet,3941


In [31]:
location_sales = df_clean.groupby("Location")["Total Spent"].sum()

location_sales

,Total Spent
Location,
In-store,760670.0
Online,791401.0


In [32]:
category_summary = df_clean.groupby("Category")["Total Spent"].agg(
    ["sum", "mean", "min", "max", "count"]
)

category_summary

,sum,mean,min,max,count
Category,,,,,
Beverages,197047.5,131.716243,5.0,410.0,1496
Butchers,208118.0,139.116310,5.0,410.0,1496
Computers and electric accessories,190692.5,129.107989,5.0,410.0,1477
Electric household essentials,203813.5,134.441623,5.0,410.0,1516
Food,194812.0,129.271400,5.0,410.0,1507
Furniture,195310.0,128.072131,5.0,410.0,1525
Milk Products,180112.0,119.042961,5.0,410.0,1513
Patisserie,182165.5,126.416031,5.0,410.0,1441


In [33]:
category_details = df_clean.groupby("Category").agg({
    "Total Spent": ["sum", "mean"],
    "Quantity": ["sum", "mean"]
})

category_details

Total Spent             Quantity          
                                           sum        mean      sum      mean
Category                                                                     
Beverages                             197047.5  131.716243   8358.0  5.586898
Butchers                              208118.0  139.116310   8206.0  5.485294
Computers and electric accessories    190692.5  129.107989   8272.0  5.600542
Electric household essentials         203813.5  134.441623   8309.0  5.480871
Food                                  194812.0  129.271400   8387.0  5.565362
Furniture                             195310.0  128.072131   8462.0  5.548852
Milk Products                         180112.0  119.042961   8339.0  5.511566
Patisserie                            182165.5  126.416031   7943.0  5.512144

In [34]:
customer_spending = (
    df_clean.groupby("Customer ID")["Total Spent"]
    .sum()
)

customer_spending.head(10)

,Total Spent
Customer ID,
CUST_01,58731.5
CUST_02,62046.5
CUST_03,60811.0
CUST_04,61767.5
CUST_05,66974.5
CUST_06,58632.5
CUST_07,60694.5
CUST_08,67351.5
CUST_09,61423.5


In [35]:
top_customers = (
    customer_spending
    .sort_values(ascending=False)
    .head(10)
)

top_customers

,Total Spent
Customer ID,
CUST_24,68452.0
CUST_08,67351.5
CUST_05,66974.5
CUST_16,65570.5
CUST_13,65037.0
CUST_23,64507.0
CUST_10,63155.5
CUST_15,63117.5
CUST_21,62933.0


In [36]:
category_location_sales = (
    df_clean
    .groupby(["Category", "Location"])["Total Spent"]
    .sum()
)

category_location_sales

Category                            Location
Beverages                           In-store     98026.0
                                    Online       99021.5
Butchers                            In-store    101777.0
                                    Online      106341.0
Computers and electric accessories  In-store     87323.5
                                    Online      103369.0
Electric household essentials       In-store     97778.5
                                    Online      106035.0
Food                                In-store     95926.0
                                    Online       98886.0
Furniture                           In-store     99611.0
                                    Online       95699.0
Milk Products                       In-store     88813.0
                                    Online       91299.0
Patisserie                          In-store     91415.0
                                    Online       90750.5
Name: Total Spent, dtype: float64

In [37]:
category_location_df = category_location_sales.reset_index()

category_location_df

,Category,Location,Total Spent
0,Beverages,In-store,98026.0
1,Beverages,Online,99021.5
2,Butchers,In-store,101777.0
3,Butchers,Online,106341.0
4,Computers and electric accessories,In-store,87323.5
5,Computers and electric accessories,Online,103369.0
6,Electric household essentials,In-store,97778.5
7,Electric household essentials,Online,106035.0
8,Food,In-store,95926.0
9,Food,Online,98886.0


In [38]:
overall_summary = df_clean.agg({
    "Total Spent": ["sum", "mean", "min", "max"],
    "Quantity": ["sum", "mean"]
})

overall_summary

,Total Spent,Quantity
sum,1.552071e+06,66276.00000
mean,1.296526e+02,5.53638
min,5.000000e+00,NaN
max,4.100000e+02,NaN


#Day-37

In [39]:
import pandas as pd

df_clean = pd.read_csv("retail_store_sales_cleaned.csv")

print("Dataset Shape:", df_clean.shape)
df_clean.head()

Dataset Shape: (11971, 11)


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,Unknown
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [40]:
category_sales = (
    df_clean.groupby("Category")["Total Spent"]
    .sum()
    .reset_index()
)

category_sales

,Category,Total Spent
0,Beverages,197047.5
1,Butchers,208118.0
2,Computers and electric accessories,190692.5
3,Electric household essentials,203813.5
4,Food,194812.0
5,Furniture,195310.0
6,Milk Products,180112.0
7,Patisserie,182165.5


In [41]:
category_quantity = (
    df_clean.groupby("Category")["Quantity"]
    .sum()
    .reset_index()
)

category_quantity

,Category,Quantity
0,Beverages,8358.0
1,Butchers,8206.0
2,Computers and electric accessories,8272.0
3,Electric household essentials,8309.0
4,Food,8387.0
5,Furniture,8462.0
6,Milk Products,8339.0
7,Patisserie,7943.0


In [42]:
category_analysis = pd.merge(
    category_sales,
    category_quantity,
    on="Category"
)

category_analysis

,Category,Total Spent,Quantity
0,Beverages,197047.5,8358.0
1,Butchers,208118.0,8206.0
2,Computers and electric accessories,190692.5,8272.0
3,Electric household essentials,203813.5,8309.0
4,Food,194812.0,8387.0
5,Furniture,195310.0,8462.0
6,Milk Products,180112.0,8339.0
7,Patisserie,182165.5,7943.0


In [43]:
category_analysis_inner = pd.merge(
    category_sales,
    category_quantity,
    on="Category",
    how="inner"
)

category_analysis_inner

,Category,Total Spent,Quantity
0,Beverages,197047.5,8358.0
1,Butchers,208118.0,8206.0
2,Computers and electric accessories,190692.5,8272.0
3,Electric household essentials,203813.5,8309.0
4,Food,194812.0,8387.0
5,Furniture,195310.0,8462.0
6,Milk Products,180112.0,8339.0
7,Patisserie,182165.5,7943.0


In [44]:
category_analysis_left = pd.merge(
    category_sales,
    category_quantity,
    on="Category",
    how="left"
)

category_analysis_left

,Category,Total Spent,Quantity
0,Beverages,197047.5,8358.0
1,Butchers,208118.0,8206.0
2,Computers and electric accessories,190692.5,8272.0
3,Electric household essentials,203813.5,8309.0
4,Food,194812.0,8387.0
5,Furniture,195310.0,8462.0
6,Milk Products,180112.0,8339.0
7,Patisserie,182165.5,7943.0


In [45]:
category_analysis_right = pd.merge(
    category_sales,
    category_quantity,
    on="Category",
    how="right"
)

category_analysis_right

,Category,Total Spent,Quantity
0,Beverages,197047.5,8358.0
1,Butchers,208118.0,8206.0
2,Computers and electric accessories,190692.5,8272.0
3,Electric household essentials,203813.5,8309.0
4,Food,194812.0,8387.0
5,Furniture,195310.0,8462.0
6,Milk Products,180112.0,8339.0
7,Patisserie,182165.5,7943.0


In [46]:
category_sales_filtered = category_sales[
    category_sales["Total Spent"] > 190000
]

category_sales_filtered


,Category,Total Spent
0,Beverages,197047.5
1,Butchers,208118.0
2,Computers and electric accessories,190692.5
3,Electric household essentials,203813.5
4,Food,194812.0
5,Furniture,195310.0


In [47]:
left_join_result = pd.merge(
    category_sales_filtered,
    category_quantity,
    on="Category",
    how="left"
)

left_join_result

,Category,Total Spent,Quantity
0,Beverages,197047.5,8358.0
1,Butchers,208118.0,8206.0
2,Computers and electric accessories,190692.5,8272.0
3,Electric household essentials,203813.5,8309.0
4,Food,194812.0,8387.0
5,Furniture,195310.0,8462.0


In [48]:
right_join_result = pd.merge(
    category_sales_filtered,
    category_quantity,
    on="Category",
    how="right"
)

right_join_result

,Category,Total Spent,Quantity
0,Beverages,197047.5,8358.0
1,Butchers,208118.0,8206.0
2,Computers and electric accessories,190692.5,8272.0
3,Electric household essentials,203813.5,8309.0
4,Food,194812.0,8387.0
5,Furniture,195310.0,8462.0
6,Milk Products,NaN,8339.0
7,Patisserie,NaN,7943.0


In [49]:
outer_join_result = pd.merge(
    category_sales_filtered,
    category_quantity,
    on="Category",
    how="outer"
)

outer_join_result

,Category,Total Spent,Quantity
0,Beverages,197047.5,8358.0
1,Butchers,208118.0,8206.0
2,Computers and electric accessories,190692.5,8272.0
3,Electric household essentials,203813.5,8309.0
4,Food,194812.0,8387.0
5,Furniture,195310.0,8462.0
6,Milk Products,NaN,8339.0
7,Patisserie,NaN,7943.0


In [50]:
category_stats = (
    df_clean.groupby("Category")["Total Spent"]
    .agg(["mean", "max"])
    .reset_index()
)

category_stats

,Category,mean,max
0,Beverages,131.716243,410.0
1,Butchers,139.116310,410.0
2,Computers and electric accessories,129.107989,410.0
3,Electric household essentials,134.441623,410.0
4,Food,129.271400,410.0
5,Furniture,128.072131,410.0
6,Milk Products,119.042961,410.0
7,Patisserie,126.416031,410.0


In [51]:
category_avg = (
    df_clean.groupby("Category")["Total Spent"]
    .mean()
    .reset_index(name="Total Spent")
)

category_max = (
    df_clean.groupby("Category")["Total Spent"]
    .max()
    .reset_index(name="Total Spent")
)

category_avg_max = pd.merge(
    category_avg,
    category_max,
    on="Category"
)

category_avg_max

,Category,Total Spent_x,Total Spent_y
0,Beverages,131.716243,410.0
1,Butchers,139.116310,410.0
2,Computers and electric accessories,129.107989,410.0
3,Electric household essentials,134.441623,410.0
4,Food,129.271400,410.0
5,Furniture,128.072131,410.0
6,Milk Products,119.042961,410.0
7,Patisserie,126.416031,410.0


In [52]:
category_avg_max = pd.merge(
    category_avg,
    category_max,
    on="Category",
    suffixes=("_Average", "_Maximum")
)

category_avg_max

,Category,Total Spent_Average,Total Spent_Maximum
0,Beverages,131.716243,410.0
1,Butchers,139.116310,410.0
2,Computers and electric accessories,129.107989,410.0
3,Electric household essentials,134.441623,410.0
4,Food,129.271400,410.0
5,Furniture,128.072131,410.0
6,Milk Products,119.042961,410.0
7,Patisserie,126.416031,410.0


In [53]:
online_data = df_clean[df_clean["Location"] == "Online"]
instore_data = df_clean[df_clean["Location"] == "In-store"]

combined_data = pd.concat(
    [online_data, instore_data],
    ignore_index=True
)

print("Original Shape:", df_clean.shape)
print("Combined Shape:", combined_data.shape)

Original Shape: (11971, 11)
Combined Shape: (11971, 11)


In [54]:
print("Online Rows:", len(online_data))
print("In-store Rows:", len(instore_data))
print("Combined Rows:", len(combined_data))

combined_data.head()

Online Rows: 6068
In-store Rows: 5903
Combined Rows: 11971


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,Unknown
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [55]:
sales_by_category = (
    df_clean.groupby("Category")["Total Spent"]
    .sum()
    .rename("Total Sales")
)

quantity_by_category = (
    df_clean.groupby("Category")["Quantity"]
    .sum()
    .rename("Total Quantity")
)

category_joined = sales_by_category.to_frame().join(
    quantity_by_category
)

category_joined

,Total Sales,Total Quantity
Category,,
Beverages,197047.5,8358.0
Butchers,208118.0,8206.0
Computers and electric accessories,190692.5,8272.0
Electric household essentials,203813.5,8309.0
Food,194812.0,8387.0
Furniture,195310.0,8462.0
Milk Products,180112.0,8339.0
Patisserie,182165.5,7943.0


In [56]:
category_summary = (
    df_clean.groupby("Category")
    .agg(
        Total_Sales=("Total Spent", "sum"),
        Total_Quantity=("Quantity", "sum"),
        Average_Sale=("Total Spent", "mean"),
        Transactions=("Transaction ID", "count")
    )
    .reset_index()
)

category_summary

,Category,Total_Sales,Total_Quantity,Average_Sale,Transactions
0,Beverages,197047.5,8358.0,131.716243,1496
1,Butchers,208118.0,8206.0,139.116310,1496
2,Computers and electric accessories,190692.5,8272.0,129.107989,1477
3,Electric household essentials,203813.5,8309.0,134.441623,1516
4,Food,194812.0,8387.0,129.271400,1507
5,Furniture,195310.0,8462.0,128.072131,1525
6,Milk Products,180112.0,8339.0,119.042961,1513
7,Patisserie,182165.5,7943.0,126.416031,1441


In [57]:
category_summary_sorted = category_summary.sort_values(
    "Total_Sales",
    ascending=False
)

category_summary_sorted

,Category,Total_Sales,Total_Quantity,Average_Sale,Transactions
1,Butchers,208118.0,8206.0,139.116310,1496
3,Electric household essentials,203813.5,8309.0,134.441623,1516
0,Beverages,197047.5,8358.0,131.716243,1496
5,Furniture,195310.0,8462.0,128.072131,1525
4,Food,194812.0,8387.0,129.271400,1507
2,Computers and electric accessories,190692.5,8272.0,129.107989,1477
7,Patisserie,182165.5,7943.0,126.416031,1441
6,Milk Products,180112.0,8339.0,119.042961,1513


In [58]:
category_summary_sorted = category_summary_sorted.reset_index(drop=True)

category_summary_sorted

,Category,Total_Sales,Total_Quantity,Average_Sale,Transactions
0,Butchers,208118.0,8206.0,139.116310,1496
1,Electric household essentials,203813.5,8309.0,134.441623,1516
2,Beverages,197047.5,8358.0,131.716243,1496
3,Furniture,195310.0,8462.0,128.072131,1525
4,Food,194812.0,8387.0,129.271400,1507
5,Computers and electric accessories,190692.5,8272.0,129.107989,1477
6,Patisserie,182165.5,7943.0,126.416031,1441
7,Milk Products,180112.0,8339.0,119.042961,1513
